# C2BM graph robustness results

This notebook compares the original C2BM run with graph topology perturbation runs. It reads the result pickles written by `main.py`, makes a side-by-side graph PDF and intervention-curve PDF for each dataset, and exports CSV tables under `outputs/reports/`.

Run from the repository root. By default, the notebook finds the newest baseline run per dataset under `outputs/multirun/` by reading its Hydra config. Set `ORIGINAL_RUNS` below to explicit run directories if you want to pin a particular baseline/seed.

In [ ]:
from pathlib import Path
import json, pickle, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

PURPLE = {"ink": "#3f315c", "main": "#6d55a3", "mid": "#9277bf", "light": "#b9a6d5", "pale": "#e9e2f2", "grid": "#ddd6e8"}
PURPLE_SERIES = [PURPLE["main"], "#8067b0", "#a084c5", "#5c4a8d", "#b59bcf"]
plt.rcParams.update({
    "font.family": "serif", "font.serif": ["STIXGeneral", "DejaVu Serif"], "mathtext.fontset": "stix",
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "axes.edgecolor": PURPLE["ink"], "axes.labelcolor": PURPLE["ink"],
    "xtick.color": PURPLE["ink"], "ytick.color": PURPLE["ink"],
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": PURPLE["grid"], "grid.linewidth": 0.6,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "pdf.fonttype": 42, "ps.fonttype": 42, "savefig.bbox": "tight",
})

_cwd = Path.cwd().resolve()
ROOT = next((p for p in (_cwd, *_cwd.parents) if (p / "learned_graphs").is_dir() and (p / "conf").is_dir()), _cwd)
REPORT_DIR = ROOT / "outputs" / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Optional: pin exact original run directories. Each directory contains results/y_accuracy.pkl.
# Example: ORIGINAL_RUNS = {"celeba": Path("outputs/multirun/2026-10-09/12-30-00/0")}
ORIGINAL_RUNS = {}
MODEL_SEED = 1
print(f"Repository: {ROOT}\nReports: {REPORT_DIR}")

## Discover runs and read saved metrics

Original runs are selected from `outputs/multirun/**/results/y_accuracy.pkl`. Topology runs are discovered from `outputs/graph_sanity/<dataset>/topology_seed_*/model_seed_<seed>/results/`; density runs from `outputs/graph_sanity/<dataset>/density_seed_*/model_seed_<seed>/results/`. The notebook keeps going when a run is incomplete and reports missing files.

In [ ]:
def load_pickle(path):
    with Path(path).open("rb") as f:
        return pickle.load(f)

def get_config_dataset(run_dir):
    cfg_path = Path(run_dir) / ".hydra" / "config.yaml"
    if not cfg_path.exists():
        return None
    try:
        import yaml
        cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8")) or {}
        ds = cfg.get("dataset", {})
        return ds.get("name") if isinstance(ds, dict) else None
    except Exception as exc:
        warnings.warn(f"Could not read {cfg_path}: {exc}")
        return None

def discover_original_run(dataset):
    explicit = ORIGINAL_RUNS.get(dataset)
    if explicit is not None:
        p = (ROOT / explicit).resolve() if not Path(explicit).is_absolute() else Path(explicit)
        if not (p / "results" / "y_accuracy.pkl").exists():
            raise FileNotFoundError(f"No results/y_accuracy.pkl in configured original run: {p}")
        return p
    candidates = []
    for metric in (ROOT / "outputs" / "multirun").glob("**/results/y_accuracy.pkl"):
        run_dir = metric.parent.parent
        if get_config_dataset(run_dir) == dataset:
            candidates.append(run_dir)
    if not candidates:
        return None
    return max(candidates, key=lambda p: (p / "results" / "y_accuracy.pkl").stat().st_mtime)

def read_metric(run_dir, filename):
    path = Path(run_dir) / "results" / filename
    return load_pickle(path) if path.exists() else None

def parse_level(key):
    match = re.search(r"level\s*(\d+)", str(key), flags=re.I)
    return int(match.group(1)) if match else None

def scalar_task_accuracy(run_dir):
    d = read_metric(run_dir, "y_accuracy.pkl")
    return None if not d else d.get("_baseline")

datasets = sorted(p.parent.name for p in (ROOT / "learned_graphs").glob("*/graph.pkl"))
original_runs = {d: discover_original_run(d) for d in datasets}
print("Baseline run selection:")
for d, p in original_runs.items():
    print(f"  {d}: {p if p else 'NOT FOUND'}")

topology_runs = {}
for dataset in datasets:
    base = ROOT / "outputs" / "graph_sanity" / dataset
    found = []
    for run_dir in base.glob("topology_seed_*/model_seed_*/"):
        if (run_dir / "results" / "y_accuracy.pkl").exists():
            m = re.search(r"topology_seed_(\d+)", str(run_dir))
            seed = int(m.group(1)) if m else -1
            found.append((seed, run_dir))
    topology_runs[dataset] = sorted(found)
    print(f"{dataset}: {len(found)} completed topology result folder(s)")

density_runs = {}
for dataset in datasets:
    base = ROOT / "outputs" / "graph_sanity" / dataset
    found = []
    for run_dir in base.glob("density_seed_*/model_seed_*/"):
        if (run_dir / "results" / "y_accuracy.pkl").exists():
            m = re.search(r"density_seed_(\d+)", str(run_dir))
            seed = int(m.group(1)) if m else -1
            found.append((seed, run_dir))
    density_runs[dataset] = sorted(found)
    print(f"{dataset}: {len(found)} completed density result folder(s)")

## Graph comparison figures

The original graph and each perturbation are drawn in one row, using the original graph’s node positions for every panel. Titles report edge count and directed adjacency Hamming distance from the learned graph; density panels also report DAG density `E / (N(N−1)/2)`. The figures use a restrained purple paper style.

In [ ]:
def load_adjacency(path):
    graph = load_pickle(path)
    if isinstance(graph, pd.DataFrame):
        return graph
    return pd.DataFrame(graph)

def to_digraph(adj):
    G = nx.DiGraph()
    G.add_nodes_from(list(adj.index))
    for source in adj.index:
        for target in adj.columns:
            if adj.loc[source, target] == 1:
                G.add_edge(source, target)
    return G

def adjacency_hamming_distance(reference, candidate):
    candidate = candidate.reindex(index=reference.index, columns=reference.columns, fill_value=0)
    return int(np.count_nonzero(reference.to_numpy() != candidate.to_numpy()))

def draw_graph_comparison(dataset, run_records, experiment="topology"):
    graph_path = ROOT / "learned_graphs" / dataset / "graph.pkl"
    if not graph_path.exists():
        warnings.warn(f"No original graph at {graph_path}")
        return
    original_adj = load_adjacency(graph_path)
    original = to_digraph(original_adj)
    records = [("Original", original, original_adj)]
    for seed, run_dir in run_records:
        pert_path = ROOT / "perturb_graph" / dataset / experiment / f"random_seed_{seed}.pkl"
        if pert_path.exists():
            pert_adj = load_adjacency(pert_path)
            records.append((f"{experiment.title()} seed {seed}", to_digraph(pert_adj), pert_adj))
        else:
            warnings.warn(f"Missing perturbation graph: {pert_path}")
    pos = nx.spring_layout(original, seed=7)
    fig, axes = plt.subplots(1, len(records), figsize=(max(6, 5 * len(records)), 5), squeeze=False)
    for index, (ax, (title, graph, adj)) in enumerate(zip(axes[0], records)):
        if set(graph.nodes) != set(original.nodes):
            warnings.warn(f"Node set differs for {title} ({dataset}); using that graph's own layout")
            graph_pos = nx.spring_layout(graph, seed=7)
        else:
            graph_pos = pos
        nx.draw_networkx_nodes(graph, graph_pos, ax=ax, node_size=820, node_color=PURPLE["pale"], edgecolors=PURPLE["ink"], linewidths=0.8)
        nx.draw_networkx_labels(graph, graph_pos, ax=ax, font_size=7, font_color=PURPLE["ink"])
        nx.draw_networkx_edges(graph, graph_pos, ax=ax, arrows=True, arrowsize=13, width=1.15,
                               edge_color=PURPLE["ink"] if index == 0 else PURPLE_SERIES[(index - 1) % len(PURPLE_SERIES)],
                               connectionstyle="arc3,rad=0.06", min_source_margin=12, min_target_margin=16)
        distance = adjacency_hamming_distance(original_adj, adj)
        max_dag_edges = len(adj) * (len(adj) - 1) / 2
        detail = f"E={graph.number_of_edges()}, dH={distance}"
        if experiment == "density":
            detail += f", DAG density={graph.number_of_edges() / max_dag_edges:.2f}"
        ax.set_title(f"{title}\n{detail}", color=PURPLE["ink"], pad=8)
        ax.axis("off")
    fig.suptitle(f"{dataset}: original and {experiment} graphs", color=PURPLE["ink"], y=1.02)
    fig.tight_layout()
    out = REPORT_DIR / f"graphs_{dataset}_original_vs_{experiment}.pdf"
    fig.savefig(out, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved {out}")

for dataset in datasets:
    draw_graph_comparison(dataset, topology_runs[dataset], "topology")

## Density perturbation experiment

The density job creates five reproducible random DAGs with distinct edge counts sampled from a DAG-density interval of 0.30–0.80. For CelebA’s 10 graph nodes this is 14–36 edges. The target remains a sink. Each graph panel reports its edge count, DAG density, and directed adjacency Hamming distance from the learned graph.

Density outputs are discovered from `outputs/graph_sanity/celeba/density_seed_<seed>/model_seed_1/`; graph matrices and JSON metadata are read from `perturb_graph/celeba/density/`.

In [ ]:
for dataset in datasets:
    if density_runs[dataset]:
        draw_graph_comparison(dataset, density_runs[dataset], "density")
    else:
        print(f"No completed density runs found for {dataset}")

density_graph_rows = []
for dataset in datasets:
    ref_path = ROOT / "learned_graphs" / dataset / "graph.pkl"
    if not ref_path.exists():
        continue
    ref_adj = load_adjacency(ref_path)
    for seed, _ in density_runs[dataset]:
        graph_path = ROOT / "perturb_graph" / dataset / "density" / f"random_seed_{seed}.pkl"
        metadata_path = graph_path.with_suffix(".json")
        if not graph_path.exists():
            continue
        adj = load_adjacency(graph_path)
        meta = json.loads(metadata_path.read_text(encoding="utf-8")) if metadata_path.exists() else {}
        density_graph_rows.append({"dataset": dataset, "density_seed": seed,
                                 "edge_count": int(adj.to_numpy().sum()),
                                 "dag_density": meta.get("perturbed_dag_density", int(adj.to_numpy().sum()) / (len(adj) * (len(adj)-1) / 2)),
                                 "adjacency_hamming_distance": adjacency_hamming_distance(ref_adj, adj),
                                 "shared_directed_edges": meta.get("shared_directed_edges")})
density_graph_table = pd.DataFrame(density_graph_rows)
density_graph_table.to_csv(REPORT_DIR / "c2bm_density_graph_metadata.csv", index=False)
display(density_graph_table)

## Accuracy tables and intervention curves

The summary includes task accuracy, mean and per-concept accuracy, edge count, graph density, and adjacency Hamming distance. Topology and density runs share the combined tables and also receive density-specific CSV exports. Intervention-level CSVs are long format for plotting later in Python or LaTeX; level 0 is the model’s no-intervention baseline pass.

In [ ]:
summary_rows = []
level_curve_rows = []
single_intervention_rows = []
level_concept_rows = []

def add_run_metrics(dataset, run_type, graph_seed, run_dir, graph_path=None):
    y_acc = read_metric(run_dir, "y_accuracy.pkl") or {}
    c_acc = read_metric(run_dir, "c_accuracy.pkl") or {}
    level_y = read_metric(run_dir, "level_interventions_on_y.pkl") or {}
    single_y = read_metric(run_dir, "single_c_interventions_on_y.pkl") or {}
    level_c = read_metric(run_dir, "level_interventions_on_c.pkl") or {}
    task_acc = y_acc.get("_baseline")
    row = {"dataset": dataset, "run_type": run_type, "topology_seed": graph_seed,
           "model_seed": MODEL_SEED, "task_accuracy": task_acc,
           "mean_concept_accuracy": float(np.mean(list(c_acc.values()))) if c_acc else np.nan,
           "run_dir": str(Path(run_dir).relative_to(ROOT)) if Path(run_dir).is_relative_to(ROOT) else str(run_dir)}
    for concept, value in c_acc.items():
        row[f"concept_accuracy::{concept}"] = value
    if graph_path and Path(graph_path).exists():
        try:
            adj = load_adjacency(graph_path)
            row["edge_count"] = int(adj.to_numpy().sum())
            max_dag_edges = len(adj) * (len(adj) - 1) / 2
            row["dag_density"] = row["edge_count"] / max_dag_edges
            reference_path = ROOT / "learned_graphs" / dataset / "graph.pkl"
            if reference_path.exists():
                row["adjacency_hamming_distance"] = adjacency_hamming_distance(load_adjacency(reference_path), adj)
            meta_path = Path(graph_path).with_suffix(".json")
            if meta_path.exists():
                meta = json.loads(meta_path.read_text(encoding="utf-8"))
                row["edge_overlap_fraction"] = meta.get("edge_overlap_fraction")
                row["shared_directed_edges"] = meta.get("shared_directed_edges")
        except Exception as exc:
            warnings.warn(f"Could not read graph metadata for {graph_path}: {exc}")
    summary_rows.append(row)
    if not level_y:
        warnings.warn(f"No level_interventions_on_y.pkl data for {dataset}/{run_type} seed={graph_seed}; run may still be in progress or the metric file may be missing")
    for key, value in level_y.items():
        level = parse_level(key)
        if level is not None:
            level_curve_rows.append({"dataset": dataset, "run_type": run_type,
                                     "topology_seed": graph_seed, "model_seed": MODEL_SEED,
                                     "level": level, "task_accuracy": value})
    for concept, value in single_y.items():
        if concept != "_baseline":
            single_intervention_rows.append({"dataset": dataset, "run_type": run_type,
                                             "topology_seed": graph_seed, "model_seed": MODEL_SEED,
                                             "intervened_concept": concept, "task_accuracy": value})
    for key, value in level_c.items():
        match = re.match(r"level\s*(\d+)/node\s*(.+)", str(key), flags=re.I)
        if match:
            level_concept_rows.append({"dataset": dataset, "run_type": run_type,
                                       "topology_seed": graph_seed, "model_seed": MODEL_SEED,
                                       "level": int(match.group(1)), "concept": match.group(2),
                                       "concept_accuracy": value})

for dataset in datasets:
    original_dir = original_runs.get(dataset)
    if original_dir:
        add_run_metrics(dataset, "original", None, original_dir, ROOT / "learned_graphs" / dataset / "graph.pkl")
    else:
        warnings.warn(f"Skipping original metrics for {dataset}: no baseline run found")
    for seed, run_dir in topology_runs[dataset]:
        graph_path = ROOT / "perturb_graph" / dataset / "topology" / f"random_seed_{seed}.pkl"
        add_run_metrics(dataset, "topology", seed, run_dir, graph_path)
    for seed, run_dir in density_runs[dataset]:
        graph_path = ROOT / "perturb_graph" / dataset / "density" / f"random_seed_{seed}.pkl"
        add_run_metrics(dataset, "density", seed, run_dir, graph_path)

summary = pd.DataFrame(summary_rows)
level_curves = pd.DataFrame(level_curve_rows, columns=["dataset", "run_type", "topology_seed", "model_seed", "level", "task_accuracy"])
single_interventions = pd.DataFrame(single_intervention_rows, columns=["dataset", "run_type", "topology_seed", "model_seed", "intervened_concept", "task_accuracy"])
level_concepts = pd.DataFrame(level_concept_rows, columns=["dataset", "run_type", "topology_seed", "model_seed", "level", "concept", "concept_accuracy"])
summary.to_csv(REPORT_DIR / "c2bm_accuracy_summary.csv", index=False)
level_curves.to_csv(REPORT_DIR / "c2bm_intervention_level_task_accuracy.csv", index=False)
single_interventions.to_csv(REPORT_DIR / "c2bm_single_concept_intervention_task_accuracy.csv", index=False)
level_concepts.to_csv(REPORT_DIR / "c2bm_intervention_level_concept_accuracy.csv", index=False)
density_summary = summary[summary.run_type == "density"].copy() if not summary.empty else summary.copy()
density_curves = level_curves[level_curves.run_type.isin(["original", "density"])].copy()
density_single = single_interventions[single_interventions.run_type == "density"].copy()
density_level_concepts = level_concepts[level_concepts.run_type == "density"].copy()
density_summary.to_csv(REPORT_DIR / "c2bm_celeba_density_accuracy_summary.csv", index=False)
density_curves.to_csv(REPORT_DIR / "c2bm_celeba_density_intervention_level_task_accuracy.csv", index=False)
density_single.to_csv(REPORT_DIR / "c2bm_celeba_density_single_concept_interventions.csv", index=False)
density_level_concepts.to_csv(REPORT_DIR / "c2bm_celeba_density_intervention_level_concept_accuracy.csv", index=False)
print(f"Saved CSV tables to {REPORT_DIR}")
display(summary)

In [ ]:
for dataset in datasets:
    for experiment in ["topology", "density"]:
        curves = level_curves[(level_curves.dataset == dataset) & (level_curves.run_type.isin(["original", experiment]))]
        if curves.empty or not (curves.run_type == experiment).any():
            print(f"No {experiment} intervention curves found for {dataset}; skipping")
            continue
        fig, ax = plt.subplots(figsize=(6.6, 4.4))
        original = curves[curves.run_type == "original"].sort_values("level")
        if not original.empty:
            ax.plot(original.level, original.task_accuracy, color=PURPLE["ink"], linestyle="--", marker="D",
                    linewidth=2.2, markersize=4, label="Original C2BM", zorder=5)
        pert = curves[curves.run_type == experiment]
        for idx, (seed, group) in enumerate(pert.groupby("topology_seed")):
            group = group.sort_values("level")
            label = f"{experiment.title()} seed {int(seed)}"
            if experiment == "density":
                meta_path = ROOT / "perturb_graph" / dataset / "density" / f"random_seed_{int(seed)}.json"
                if meta_path.exists():
                    rho = json.loads(meta_path.read_text(encoding="utf-8")).get("perturbed_dag_density")
                    if rho is not None:
                        label += f" ($\\rho$={rho:.2f})"
            ax.plot(group.level, group.task_accuracy, color=PURPLE_SERIES[idx % len(PURPLE_SERIES)],
                    marker="o", markersize=3.8, linewidth=1.5, label=label)
        ax.set(title=f"{dataset}: task accuracy under {experiment} interventions",
               xlabel="Intervention policy level", ylabel="Task accuracy")
        ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.8)
        ax.legend(frameon=False, ncol=2)
        fig.tight_layout()
        out_name = f"density_intervention_curve_{dataset}.pdf" if experiment == "density" else f"intervention_curve_{dataset}.pdf"
        out = REPORT_DIR / out_name
        fig.savefig(out, bbox_inches="tight", metadata={"Title": f"{dataset} {experiment} intervention curve"})
        plt.show()
        plt.close(fig)
        print(f"Saved {out}")

display(level_curves.sort_values(["dataset", "run_type", "topology_seed", "level"]))
display(single_interventions.sort_values(["dataset", "run_type", "topology_seed", "intervened_concept"]))
display(level_concepts.sort_values(["dataset", "run_type", "topology_seed", "level", "concept"]))

## Files created

- `outputs/reports/graphs_<dataset>_original_vs_topology.pdf`
- `outputs/reports/graphs_<dataset>_original_vs_density.pdf`
- `outputs/reports/intervention_curve_<dataset>.pdf`
- `outputs/reports/density_intervention_curve_<dataset>.pdf`
- `outputs/reports/c2bm_accuracy_summary.csv`
- `outputs/reports/c2bm_density_graph_metadata.csv`
- `outputs/reports/c2bm_celeba_density_accuracy_summary.csv`
- `outputs/reports/c2bm_celeba_density_intervention_level_task_accuracy.csv`
- `outputs/reports/c2bm_celeba_density_single_concept_interventions.csv`
- `outputs/reports/c2bm_celeba_density_intervention_level_concept_accuracy.csv`
- `outputs/reports/c2bm_intervention_level_task_accuracy.csv`
- `outputs/reports/c2bm_single_concept_intervention_task_accuracy.csv`
- `outputs/reports/c2bm_intervention_level_concept_accuracy.csv`

The notebook uses `level_interventions_on_y.pkl` for the level curve and `single_c_interventions_on_y.pkl` for per-concept task accuracy after intervention. Those are aggregate test-set accuracy values; the repository does not save per-example predictions in these files.

## Debug raw result pickles

Choose a run below to print the result files exactly as stored. This helps distinguish a missing file, an empty dictionary, missing concept keys, and values that are actually NaN. Run the discovery and helper cells above first.

In [ ]:
from pprint import pprint

DEBUG_DATASET = "celeba"
DEBUG_RUN_TYPE = "topology"  # "topology" or "original"
DEBUG_TOPOLOGY_SEED = 2

if DEBUG_RUN_TYPE == "original":
    debug_run_dir = original_runs.get(DEBUG_DATASET)
elif DEBUG_RUN_TYPE == "topology":
    matches = [run_dir for seed, run_dir in topology_runs.get(DEBUG_DATASET, []) if seed == DEBUG_TOPOLOGY_SEED]
    debug_run_dir = matches[0] if matches else None
else:
    raise ValueError("DEBUG_RUN_TYPE must be 'original' or 'topology'")

if debug_run_dir is None:
    print("No matching run found. Check DEBUG_DATASET, DEBUG_RUN_TYPE, and DEBUG_TOPOLOGY_SEED.")
else:
    debug_results_dir = Path(debug_run_dir) / "results"
    print(f"Run directory: {debug_run_dir}")
    print(f"Results directory exists: {debug_results_dir.is_dir()}")
    for filename in ["y_accuracy.pkl", "c_accuracy.pkl", "single_c_interventions_on_y.pkl", "level_interventions_on_y.pkl", "level_interventions_on_c.pkl"]:
        path = debug_results_dir / filename
        print(f"\n--- {filename} ---")
        if not path.exists():
            print("MISSING")
            continue
        print(f"Path: {path} | Size: {path.stat().st_size:,} bytes")
        try:
            value = load_pickle(path)
            print(f"Type: {type(value).__name__}")
            pprint(value, sort_dicts=False)
            if isinstance(value, dict):
                nan_keys = [k for k, v in value.items() if isinstance(v, (int, float, np.number)) and not np.isfinite(v)]
                print(f"Non-finite numeric values: {nan_keys}")
        except Exception as exc:
            print(f"Could not load pickle: {type(exc).__name__}: {exc}")